In [1]:
import pandas as pd

raw = pd.read_csv("../../data/raw/regular_season_raw.csv", dtype={"athlete_id": "string"})
roster_df = pd.read_csv("../../data/processed/bigwest_roster.csv", dtype={"athlete_id": "string"})
bw_results = pd.read_csv("../../data/processed/bigwest_2019_2026_results.csv", encoding="utf-8-sig")
def clean_ids(s):
    return s.astype("string").str.strip().str.replace(r"\.0$", "", regex=True)
to_scrape = []
print(len(to_scrape), "roster athletes not scraped yet")

0 roster athletes not scraped yet


## Cleaning Data

In [2]:
df = pd.read_csv("../../data/raw/regular_season_raw.csv")

# Rename columns
df = df.rename(columns={
    "athete_name": "athlete_name",
    "is_track_field": "is_track"
})

# Clean whitespace
for col in df.select_dtypes(include=["object", "string"]).columns:
    df[col] = (
        df[col].astype("string")
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .replace("", pd.NA)
    )

# Preserve original marks for extraction
original_mark = df["mark"].astype("string")

# Extract year, including dates like "Apr 13-16, 2022"
df["year"] = pd.to_numeric(
    df["meet_date"].astype("string").str.extract(r"\b(\d{4})\b")[0],
    errors="coerce"
).astype("Int64")

# Extract meters from marks like 6.72m(0.2)22' 0.75"
meters = pd.to_numeric(
    original_mark.str.extract(r"(\d+(?:\.\d+)?)\s*m(?![a-z])")[0],
    errors="coerce"
)

# Extract wind
df["wind"] = pd.to_numeric(
    original_mark.str.extract(
        r"^\s*\d+(?:\.\d+)?\s*m?\s*\(\s*([+-]?\d+(?:\.\d+)?)\s*\)"
    )[0],
    errors="coerce"
)

# Read running times in seconds
seconds = pd.to_numeric(df["mark_seconds"], errors="coerce")

# Match event labels without changing the original event column
event = (
    df["event"].astype("string")
    .str.upper()
    .str.replace(r"\s+", "", regex=True)
)

field_event = event.isin([
    "LJ", "TJ", "HJ", "PV", "SP", "DT", "HT", "JT", "WT"
])

running_event = event.isin([
    "60", "100", "200", "400", "800", "1000", "1500",
    "MILE", "3000", "5000", "10000",
    "60H", "100H", "110H", "400H",'4x440y', '600y', '300', '2 Mile', 'SMR', '4x200', '500', '300H',
    "2000S", "3000S", "4X100", "4X400", "4X800", "DMR","10,000","4x440y","600","2000",

])

points_event = event.isin([
    "DEC", "HEP", "PENT", "DECATHLON", "HEPTATHLON", "PENTATHLON","Dec","Hep"
])

# Combine performances into one mark column
df["mark"] = original_mark.str.replace(
    r"\(\s*[+-]?\d+(?:\.\d+)?\s*\)", "", regex=True
).str.strip()
df["mark_unit"] = pd.Series(pd.NA, index=df.index, dtype="string")

df.loc[seconds.notna() | running_event, "mark_unit"] = "time"

df.loc[meters.notna(), "mark"] = meters[meters.notna()].astype("string")
df.loc[meters.notna() | field_event, "mark_unit"] = "meters"

# Combined events use points
points = pd.to_numeric(
    original_mark.str.replace(",", "", regex=False),
    errors="coerce"
)
df.loc[points_event, "mark"] = points[points_event].astype("string")
df.loc[points_event, "mark_unit"] = "points"

# Preserve non-performance statuses
status = original_mark.str.strip().str.upper().str.extract(
    r"^(DNS|DNF|DQ|FS|FOUL|NM|NH|ND|NP|NT|SCR)", expand=False)
is_status = status.notna()
df.loc[is_status, "mark"] = status[is_status]

# Extract numeric placement
df["place"] = pd.to_numeric(
    df["place"].astype("string").str.extract(r"^(\d+)")[0],
    errors="coerce"
).astype("Int64")

# Select final columns
df = df[[
    "athlete_id",
    "athlete_name",
    "team",
    "meet_name",
    "meet_date",
    "year",
    "event",
    "mark",
    "mark_unit",
    "place",
    "is_track",
    "wind"
]].copy()

df = df[df["is_track"] == True].copy()

df["event_type"] = "unknown"

df.loc[running_event.reindex(df.index, fill_value=False), "event_type"] = "track"
df.loc[field_event.reindex(df.index, fill_value=False), "event_type"] = "field"
df.loc[points_event.reindex(df.index, fill_value=False), "event_type"] = "combined"
df.head()

,athlete_id,athlete_name,team,meet_name,meet_date,year,event,mark,mark_unit,place,is_track,wind,event_type
0,7657050,Benjamin Goodwin,UC DAVIS,62nd Annual Mt. SAC Relays (Collegiate),"Apr 13-16, 2022",2022,100,10.74,time,58,True,1.9,track
1,7657050,Benjamin Goodwin,UC DAVIS,62nd Annual Mt. SAC Relays (Collegiate),"Apr 13-16, 2022",2022,4x100,41.06,time,4,True,<NA>,track
2,7657050,Benjamin Goodwin,UC DAVIS,Brutus Hamilton Invitational,"Apr 9, 2022",2022,100,10.95,time,10,True,-1.1,track
3,7657050,Benjamin Goodwin,UC DAVIS,Rafer Johnson & Jackie Joyner- Kersee Invite,"Mar 26, 2022",2022,100,10.88,time,12,True,-0.4,track
4,7657050,Benjamin Goodwin,UC DAVIS,Rafer Johnson & Jackie Joyner- Kersee Invite,"Mar 26, 2022",2022,4x100,40.51,time,1,True,<NA>,track


## Code used to fix the team column so its only bigwest schools and not school from transferred


In [3]:
roster_df = pd.read_csv("../../data/processed/bigwest_roster.csv")
results_df = df.copy()
# Standardize athlete IDs and years
for data in [roster_df, results_df]:
    data["athlete_id"] = clean_ids(data["athlete_id"])
    data["year"] = pd.to_numeric(data["year"], errors="coerce").astype("Int64")

# Keep only the columns needed for matching
roster_lookup = roster_df[
    ["athlete_id", "year", "school"]
].drop_duplicates()

roster_lookup = roster_lookup[
    ~(
        (
            (roster_lookup["athlete_id"].astype(str) == "8248590") &
            (roster_lookup["year"] == 2025) &
            (roster_lookup["school"] == "LONG BEACH ST.")
        )
        |
        (
            (roster_lookup["athlete_id"].astype(str) == "8669358") &
            (roster_lookup["year"] == 2025) &
            (roster_lookup["school"] == "UC RIVERSIDE")
        )
    )
].copy()

roster_lookup.head()


,athlete_id,year,school
0,6996283,2019,CSUN
1,6996285,2019,CSUN
2,6064612,2019,CSUN
3,6996284,2019,CSUN
4,6064624,2019,CSUN


In [4]:
duplicates = roster_lookup[
    roster_lookup.duplicated(
        subset=["athlete_id", "year"],
        keep=False
    )
]

print("Conflicting roster records:", len(duplicates))

duplicates.sort_values(["athlete_id", "year"])

Conflicting roster records: 0


,athlete_id,year,school


In [5]:
print(bw_results.shape)
print(bw_results.columns.tolist())

(5257, 14)
['Unnamed: 0', 'athlete_id', 'athlete_name', 'team', 'meet_name', 'meet_year', 'gender', 'event', 'mark', 'mark_unit', 'place', 'wind', 'event_type', 'points']


In [6]:
bw_teams = (bw_results.assign(athlete_id=clean_ids(bw_results["athlete_id"]),
                              year=bw_results["meet_year"].astype("Int64"),
                              bw_team=bw_results["team"].str.upper().str.strip())
            [["athlete_id", "year", "bw_team"]].dropna().drop_duplicates(["athlete_id", "year"]))

results_updated = (results_df
                   .merge(roster_lookup, on=["athlete_id", "year"], how="left", validate="many_to_one")
                   .merge(bw_teams, on=["athlete_id", "year"], how="left", validate="many_to_one"))
results_updated["original_team"] = results_updated["team"]
results_updated["team"] = results_updated["school"].fillna(results_updated["bw_team"])
print("rows with no Big West team (dropped):", results_updated["team"].isna().sum())
results_updated = results_updated.dropna(subset=["team"]).drop(columns=["school", "bw_team"])

rows with no Big West team (dropped): 21015


In [7]:
changed = results_updated[
    results_updated["team"] != results_updated["original_team"]
]

print("Total performances:", len(results_updated))
print("Performances with corrected teams:", len(changed))
print("Unique athletes affected:", changed["athlete_id"].nunique())

changed[
    ["athlete_id", "year", "original_team", "team"]
].drop_duplicates().head(20)
changed

Total performances: 68093
Performances with corrected teams: 3344
Unique athletes affected: 113


,athlete_id,athlete_name,team,meet_name,meet_date,year,event,mark,mark_unit,place,is_track,wind,event_type,original_team
41,7149403,Matthew Okonkwo,UC DAVIS,NCAA West Preliminary Round,"May 25-28, 2022",2022,200,21.07,time,33,True,2.0,track,USC
42,7149403,Matthew Okonkwo,UC DAVIS,Big West Track Field Championships,"May 13-14, 2022",2022,200,21.08,time,1,True,2.7,track,USC
43,7149403,Matthew Okonkwo,UC DAVIS,Big West Track Field Championships,"May 13-14, 2022",2022,200,21.45,time,6,True,-0.8,track,USC
44,7149403,Matthew Okonkwo,UC DAVIS,Big West Track Field Championships,"May 13-14, 2022",2022,400,47.20,time,3,True,<NA>,track,USC
45,7149403,Matthew Okonkwo,UC DAVIS,Big West Track Field Championships,"May 13-14, 2022",2022,400,47.59,time,3,True,<NA>,track,USC
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
88635,8308467,Isabel Peterson,HAWAII,2023 MPSF Indoor Track & Field Championships,"Feb 17-18, 2023",2023,DMR,12:18.39,time,8,True,<NA>,track,CINCINNATI
88636,8308467,Isabel Peterson,HAWAII,2023 WSU Open,"Feb 2- 4, 2023",2023,60,8.09,time,50,True,<NA>,track,CINCINNATI
88637,8308467,Isabel Peterson,HAWAII,2023 WSU Open,"Feb 2- 4, 2023",2023,200,25.91,time,54,True,<NA>,track,CINCINNATI
88638,8308467,Isabel Peterson,HAWAII,2023 WSU Open,"Feb 2- 4, 2023",2023,400,58.32,time,22,True,<NA>,track,CINCINNATI


In [8]:
##Clean so everything before 2019 is deleted
results_updated = results_updated[results_updated["year"]>=2019]


## Final cleaning make it so it actually represents regular season performances not evrythign after or before

In [9]:
import re

def parse_meet_date(date_string):
    if pd.isna(date_string):
        return pd.NaT

    date_string = str(date_string).strip()

    # Extract first month, first day, and year
    match = re.match(
        r"^([A-Za-z]+)\s+(\d{1,2}).*?,\s*(\d{4})$",
        date_string
    )

    if match:
        month, day, year = match.groups()

        return pd.to_datetime(
            f"{month} {day}, {year}",
            errors="coerce"
        )

    return pd.to_datetime(date_string, errors="coerce")


# Apply the corrected function
df = results_updated.copy()
df["meet_start_date"] = df["meet_date"].apply(parse_meet_date)

# Check remaining failures
print(
    "Dates that couldn't be parsed:",
    df["meet_start_date"].isna().sum()
)

Dates that couldn't be parsed: 0


In [10]:
df = results_updated.copy()
df["meet_start_date"] = df["meet_date"].apply(parse_meet_date)
# Convert year to integer
df["year"] = pd.to_numeric(
    df["year"], errors="coerce"
).astype("Int64")

# Championship final dates (fill in verified dates for each year)
championship_end_dates = {
    2026: pd.Timestamp("2026-05-16"),
    2025: pd.Timestamp("2025-05-17"),
    2024: pd.Timestamp("2024-05-11"),
    2023: pd.Timestamp("2023-05-13"),
    2022: pd.Timestamp("2022-05-14"),
    2021: pd.Timestamp("2021-05-15"),
    2019: pd.Timestamp("2019-05-11"),
}

df["championship_end_date"] = df["year"].map(
    championship_end_dates
)

# Only January through May
valid_month = df["meet_start_date"].dt.month.between(1, 5)

# Identify Big West meets
is_big_west = df["meet_name"].str.contains(
    r"big\s*west",
    case=False,
    na=False,
    regex=True
)

# Identify Big West Challenge Cup
is_challenge_cup = df["meet_name"].str.contains(
    r"big\s*west\s*challenge\s*cup",
    case=False,
    na=False,
    regex=True
)

# Remove Big West meets EXCEPT the Challenge Cup
not_big_west = ~is_big_west | is_challenge_cup


# Remove meets starting after championship ends
before_cutoff = (
    df["meet_start_date"] <= df["championship_end_date"]
)

# Apply filters
regular_season_clean = df[
    valid_month &
    not_big_west &
    before_cutoff
].copy()

# Remove temporary column
regular_season_clean.drop(
    columns=["championship_end_date"],
    inplace=True
)

print("Original rows:", len(df))
print("Remaining rows:", len(regular_season_clean))
print("Removed rows:", len(df) - len(regular_season_clean))

Original rows: 68093
Remaining rows: 57334
Removed rows: 10759


In [11]:
bigwest_ay = (bw_results.assign(athlete_id=clean_ids(bw_results["athlete_id"]),
                                year=bw_results["meet_year"].astype("Int64"))
              [["athlete_id", "year"]].dropna().drop_duplicates())

In [12]:
reg_ay = (regular_season_clean.assign(athlete_id=clean_ids(regular_season_clean["athlete_id"]))
          [["athlete_id", "year"]].drop_duplicates())
comparison = bigwest_ay.merge(reg_ay, on=["athlete_id", "year"], how="left", indicator=True)
missing_athletes = comparison[comparison["_merge"] == "left_only"].drop(columns="_merge")
print("Big West athlete-years:", len(bigwest_ay), "| missing from regular season:", len(missing_athletes))
missing_athletes

Big West athlete-years: 3297 | missing from regular season: 2


,athlete_id,year
1336,7687512,2024
2889,5958341,2019


In [13]:
g_roster = (roster_df.assign(athlete_id=clean_ids(roster_df["athlete_id"]))
            .dropna(subset=["gender"]).drop_duplicates("athlete_id").set_index("athlete_id")["gender"])
g_bw = (bw_results.assign(athlete_id=clean_ids(bw_results["athlete_id"]))
        .dropna(subset=["athlete_id"]).drop_duplicates("athlete_id").set_index("athlete_id")["gender"])
gender_lookup = g_roster.combine_first(g_bw)

regular_season_clean["gender"] = clean_ids(regular_season_clean["athlete_id"]).map(gender_lookup)
print(regular_season_clean["gender"].value_counts(dropna=False))   # NaN should be 0

gender
women    32654
men      24680
Name: count, dtype: int64


In [14]:
def parse_mark(raw, unit=None):
    """
    Turn one raw mark string into a number in natural units.

    Args:
        raw: mark as scraped, e.g. "10.52", "1:52.31", "6.45m", "21-04.5", "5,432", "DNF".
        unit: optional unit column value (not needed; kept for compatibility).
    Returns:
        float: seconds for track, meters for field, points for multis; NaN if it can't be parsed
        (status codes such as "DNF" are handled by _prep, not here).
    """
    if raw is None or (isinstance(raw, float) and np.isnan(raw)):
        return np.nan
    metric = re.match(r"^\s*(\d+(?:\.\d+)?)\s*m(?![a-z])", str(raw), flags=re.IGNORECASE)
    if metric:                                                   # '14.21m46\' 7.5"' -> 14.21
        return float(metric.group(1))
    s = re.sub(r"[A-Za-z\"]+$", "", str(raw).strip()).strip()   # drop trailing 'm', 'w', 'h', ...
    s = s.replace(",", "")                                       # multi-event scores like "5,432"
    ft_in = re.match(r"^(\d+)\s*[-']\s*(\d+(?:\.\d+)?)$", s)
    if ft_in:
        return (float(ft_in.group(1)) * 12 + float(ft_in.group(2))) * 0.0254
    if ":" in s:
        try:
            parts = [float(x) for x in s.split(":")]
        except ValueError:
            return np.nan
        return sum(v * 60 ** (len(parts) - 1 - i) for i, v in enumerate(parts))
    try:
        return float(s)
    except ValueError:
        return np.nan

In [15]:
import numpy as np

# How far (in %) a mark can be from the athlete's own median / the event median before it's flagged
ATHLETE_TOL = {"sprint": 8, "distance": 12, "field": 20, "multi": 25, "relay": 8}
EVENT_TOL   = {"sprint": 20, "distance": 30, "field": 40, "multi": 40, "relay": 15}

def mark_group(e):
    e = str(e).upper()
    if e.startswith("4X") or e in {"DMR", "SMR"}: return "relay"
    if e in {"DEC", "HEP", "PENT"}: return "multi"
    if e in {"LJ", "TJ", "HJ", "PV", "SP", "DT", "HT", "JT", "WT"}: return "field"
    if e in {"55", "60", "100", "200", "300", "400", "55H", "60H", "100H", "110H", "300H", "400H"}: return "sprint"
    return "distance"

rs = regular_season_clean.copy()
rs["value"] = rs["mark"].map(parse_mark)
rs["group"] = rs["event"].map(mark_group)
d = pd.to_datetime(rs["meet_start_date"], errors="coerce")
rs["season"] = d.dt.year + (d.dt.month >= 8).astype(int)      # same season rule as the model
v = rs[rs["value"].notna() & (rs["value"] > 0)].copy()

# 1. far from the athlete's own median in that event and season (needs 3+ marks)
grp = v.groupby(["athlete_id", "event", "season"])["value"]
v["n_marks"] = grp.transform("size")
v["athlete_dev_pct"] = 100 * (v["value"] / grp.transform("median") - 1)
# 2. far from the event median for that gender
v["event_dev_pct"] = 100 * (v["value"] / v.groupby(["gender", "event"])["value"].transform("median") - 1)

flag_athlete = (v["n_marks"] >= 3) & (v["athlete_dev_pct"].abs() > v["group"].map(ATHLETE_TOL))
flag_event = v["event_dev_pct"].abs() > v["group"].map(EVENT_TOL)
v["flag"] = np.select([flag_athlete & flag_event, flag_athlete, flag_event],
                      ["both", "athlete", "event"], "")

suspects = (v[v["flag"] != ""]
            .sort_values("event_dev_pct", key=abs, ascending=False)
            [["athlete_id", "athlete_name", "gender", "event", "meet_name", "meet_start_date",
              "mark", "value", "n_marks", "athlete_dev_pct", "event_dev_pct", "flag"]]
            .round(1))
print(len(suspects), "suspect marks")
print(suspects["flag"].value_counts())
suspects

387 suspect marks
flag
event      221
athlete    106
both        60
Name: count, dtype: int64


C:\Users\12094\AppData\Local\Temp\ipykernel_9676\3895257605.py:38: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  .round(1))


,athlete_id,athlete_name,gender,event,meet_name,meet_start_date,mark,value,n_marks,athlete_dev_pct,event_dev_pct,flag
23452,8362406,Maleik Pabon,men,60,2026 MPSF Indoor TF Championships,2026-03-01,38.42,38.4,4,468.3,455.2,both
14409,8019949,Reyna Johnson,women,100H,The Vince O'Boyle Track and Field Classic,2022-04-01,33.55,33.6,8,136.9,125.5,both
75154,7107070,Giovanni Giordani,men,JT,UCSB vs. UC Riverside vs. Cal Poly,2019-03-09,0.03,0.0,3,-99.9,-99.9,both
64373,7111101,Tyler Nelson,men,DT,UCSB vs. UC Riverside vs. Cal Poly,2019-03-09,0.03,0.0,5,-99.9,-99.9,both
75290,6817838,Brook Surafel,men,TJ,UCSB vs. UC Riverside vs. Cal Poly,2019-03-09,0.08,0.1,4,-99.4,-99.4,both
...,...,...,...,...,...,...,...,...,...,...,...,...
397,7027157,Parris Samaniego,men,4x400,2022 Challenge Cup,2022-04-08,3:25.85,205.8,6,8.6,3.6,athlete
394,7027157,Parris Samaniego,men,4x400,Battle in the OC,2022-04-22,3:24.84,204.8,6,8.1,3.1,athlete
1573,7588491,Matthew Molina,men,110H,UCSB Invite Late Afternoon Combined Events,2022-03-09,15.22,15.2,3,-13.8,2.1,athlete
52981,7753208,Catherine Dahlberg,women,4x400,OC Dual,2021-04-22,3:58.04,238.0,3,-8.1,1.9,athlete


In [18]:
# Marks confirmed as errors: (athlete_id, event, meet_name, mark)
DROP = pd.DataFrame([
    # placeholder marks
    ("7107070", "JT",   "UCSB vs. UC Riverside vs. Cal Poly", 0.03),
    ("7111101", "DT",   "UCSB vs. UC Riverside vs. Cal Poly", 0.03),
    ("6817838", "TJ",   "UCSB vs. UC Riverside vs. Cal Poly", 0.08),
    ("5624836", "TJ",   "UCSB vs. UC Riverside vs. Cal Poly", 0.08),
    ("7055591", "LJ",   "UCSB vs. UC Riverside vs. Cal Poly", 0.08),
    ("7152232", "LJ",   "UCSB vs. UC Riverside vs. Cal Poly", 0.08),
    ("6134234", "PV",   "UCSB vs. UC Riverside vs. Cal Poly", 0.07),
    ("6630698", "PV",   "UCSB vs. UC Riverside vs. Cal Poly", 0.07),
    ("7107070", "PV",   "UCSB vs. UC Riverside vs. Cal Poly", 0.07),
    ("7110105", "PV",   "UCSB vs. UC Riverside vs. Cal Poly", 0.07),
    ("6631077", "PV",   "UCSB vs. UC Riverside vs. Cal Poly", 0.07),
    ("6666105", "PV",   "UCSB vs. UC Riverside vs. Cal Poly", 0.07),
    ("6110692", "HJ",   "UCSB vs. UC Riverside vs. Cal Poly", 0.07),
    ("8694299", "TJ",   "Mike Fanelli Track Classic", 0.11),
    # wrong event label
    ("8362406", "60",   "2026 MPSF Indoor TF Championships", 38.42),
    ("8003935", "100",  "2023 Chris Rinne Twilight", 19.70),
    ("6631077", "100",  "Cal Poly 4-Way", 21.42),
    ("7767578", "200",  "UC-Riverside Distance Invite", 34.85),
    ("8445216", "LJ",   "Mustang Invitational", 2.12),
    ("7515268", "LJ",   "MPSF Championships", 2.41),
    ("8360077", "HT",   "2023 Challenge Cup Classic", 13.86),
    ("8080815", "DT",   "Titan Friday Night Lights", 14.21),
    ("8998438", "JT",   "2025 97th Clyde Littlefield Texas Relays", 13.16),
    ("8986968", "JT",   "Challenge Cup", 13.52),
    # impossible hurdle times
    ("8019949", "100H", "The Vince O'Boyle Track and Field Classic", 33.55),
    ("7735475", "100H", "Stanford Invitational", 26.60),
    ("8986969", "100H", "Challenge Cup", 23.27),
    # other
    ("7767603", "SP",   "Lumberjack Team Challenge", 5.52),
    ("8739859", "DEC",  "65th Annual Mt. SAC Relays", 821),
], columns=["athlete_id", "event", "meet_name", "value"])

KEY = ["athlete_id", "event", "meet_name", "value"]
rc = regular_season_clean
keys = pd.DataFrame({
    "athlete_id": clean_ids(rc["athlete_id"]).to_numpy(),
    "event":      rc["event"].astype("string").str.upper().str.strip().to_numpy(),
    "meet_name":  rc["meet_name"].astype("string").str.strip().to_numpy(),
    "value":      rc["mark"].map(parse_mark).round(2).to_numpy(),
})
DROP["value"] = DROP["value"].round(2)

# Check: every DROP entry should match at least one row
check = DROP.merge(keys.drop_duplicates(), on=KEY, how="left", indicator=True)
print("unmatched:", (check["_merge"] == "left_only").sum())   # should be 0
print(check[check["_merge"] == "left_only"])

# Drop
is_bad = (keys.merge(DROP.assign(_bad=True), on=KEY, how="left")["_bad"]
              .fillna(False).astype(bool).to_numpy())
print(is_bad.sum(), "rows dropped")
regular_season_clean = rc[~is_bad].copy()

unmatched: 0
Empty DataFrame
Columns: [athlete_id, event, meet_name, value, _merge]
Index: []
29 rows dropped


In [ ]:
regular_season_clean = regular_season_clean.drop_duplicates()
regular_season_clean.to_csv(
    "../data/processed/regular_season.csv",
    index=False
)


In [ ]:
print(regular_season_clean.groupby(["year", "gender"]).size().unstack())
print(regular_season_clean["athlete_id"].nunique(), "athletes")   # about 2,900, up from about 1,500

In [ ]:
regular_season_clean.groupby(["team", "year"])["athlete_id"].nunique()